# Case Study : Stemming Vs. Lemmitatization in Sentiment Analysis of Movie Reviews

## Objective:

* To compare the effectiveness of Stemming(Using Porter Stemmer)
* Lemmatization(using WordNetLemmatizer) in preprocessing text data
* Sentiment classification, evaluating their impact on model accuracy.

In [73]:
# Import and Load Data

import nltk
import random
import pandas as pd

from nltk.corpus import movie_reviews #Dataset containing labeled movie reviews
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import re

In [74]:
nltk.download("movie_reviews")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\C5144878\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\C5144878\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\C5144878\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\C5144878\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [75]:
movie_reviews

<CategorizedPlaintextCorpusReader in 'C:\\Users\\C5144878\\AppData\\Roaming\\nltk_data\\corpora\\movie_reviews'>

In [76]:
#Load and shuffle 100 positive and 100 negative reviews

docs = [(movie_reviews.raw(fileid), category)
        for category in movie_reviews.categories()
        for fileid in movie_reviews.fileids(category)]

In [77]:
random.shuffle(docs)
docs = docs[:200] #First 200 reviews

#Create DataFrame

df = pd.DataFrame(docs, columns=["review", "label"])
df["label"] = df["label"].map({"pos":1, "neg":0})

In [78]:
df.head()

,review,label
0,"originally titled 'don't lose your head' , thi...",1
1,produced by robert lantos & stephen j . roth \...,1
2,if you haven't plunked down your hard-earned m...,0
3,the army comedy genre has never turned out a t...,0
4,i wish i could accurately describe the theme m...,0


In [79]:
df["label"].value_counts()

label
0    101
1     99
Name: count, dtype: int64

## Text Cleanning

In [80]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^\w\s]","",text)

    return text

df["clean"] = df["review"].apply(clean_text)

In [81]:
df.head()

,review,label,clean
0,"originally titled 'don't lose your head' , thi...",1,originally titled dont lose your head this pa...
1,produced by robert lantos & stephen j . roth \...,1,produced by robert lantos stephen j roth \nd...
2,if you haven't plunked down your hard-earned m...,0,if you havent plunked down your hardearned mon...
3,the army comedy genre has never turned out a t...,0,the army comedy genre has never turned out a t...
4,i wish i could accurately describe the theme m...,0,i wish i could accurately describe the theme m...


## Preprocessing Functions

In [82]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    

In [83]:
# Stopwords

stop_words = set(stopwords.words("english")) - {"not","no","never"}

In [84]:
#Stemming

stemmer = PorterStemmer()

def stem_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    return " ".join(stemmed)

#stemming process to text


In [85]:
df["stemmed"] = df["clean"].apply(stem_pipeline)

In [86]:
df["stemmed"].head()

0    origin titl dont lose head parodi scarlet pimp...
1    produc robert lanto stephen j roth direct doug...
2    havent plunk hardearn money yet wild wild west...
3    armi comedi genr never turn truli good movi do...
4    wish could accur describ theme music part 3 be...
Name: stemmed, dtype: object

## Lemmatization

In [87]:
lemmatizer = WordNetLemmatizer()

def lemma_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(filtered)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in pos_tags]
    return " ".join(lemmatized)

#Define function for lemmatizing process

In [88]:
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\C5144878\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [89]:
df["lemmatized"] = df["clean"].apply(lemma_pipeline)

In [90]:
df["lemmatized"].head()

0    originally title dont lose head parody scarlet...
1    produce robert lantos stephen j roth direct do...
2    havent plunk hardearned money yet wild wild we...
3    army comedy genre never turn truly good movie ...
4    wish could accurately describe theme music par...
Name: lemmatized, dtype: object

## Classification and Accuracy

In [91]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

#Converts text to numerical TF-IDF feature vectors for machine learnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [92]:
#Prepare data

X_stem = df["stemmed"]
X_lemma = df["lemmatized"]
y = df["label"]

In [93]:
X_stem.head()

0    origin titl dont lose head parodi scarlet pimp...
1    produc robert lanto stephen j roth direct doug...
2    havent plunk hardearn money yet wild wild west...
3    armi comedi genr never turn truli good movi do...
4    wish could accur describ theme music part 3 be...
Name: stemmed, dtype: object

In [94]:
X_lemma.head()

0    originally title dont lose head parody scarlet...
1    produce robert lantos stephen j roth direct do...
2    havent plunk hardearned money yet wild wild we...
3    army comedy genre never turn truly good movie ...
4    wish could accurately describe theme music par...
Name: lemmatized, dtype: object

In [95]:
y.head()

0    1
1    1
2    0
3    0
4    0
Name: label, dtype: int64

In [96]:
#stemming data split

X_train_s, X_test_s, y_train,y_test = train_test_split(X_stem, y, test_size = 0.2, random_state = 42)

In [97]:
X_train_l, X_test_l, _,_ = train_test_split(X_lemma, y, test_size = 0.2, random_state = 42)

In [98]:
X_train_s.shape

(160,)

In [99]:
y_train.shape

(160,)

In [100]:
vectorizer = TfidfVectorizer()
X_train_s_vec = vectorizer.fit_transform(X_train_s)
X_test_s_vec = vectorizer.transform(X_test_s)

In [101]:
X_train_l_vec = vectorizer.fit_transform(X_train_l)
X_test_l_vec = vectorizer.transform(X_test_l)

In [102]:
#Train and Evaluate

model_s = LogisticRegression(max_iter=1000)
model_s.fit(X_train_s_vec, y_train)

y_pred_s = model_s.predict(X_test_s_vec)

acc_s = accuracy_score(y_test, y_pred_s)

print("Stemming Accuracy:", round(acc_s,2))


Stemming Accuracy: 0.68


In [103]:
#Train and Evaluate

model_l = LogisticRegression(max_iter=1000)
model_l.fit(X_train_l_vec, y_train)

y_pred_l = model_l.predict(X_test_l_vec)

acc_l = accuracy_score(y_test, y_pred_l)

print("Stemming Accuracy:", round(acc_l,2))


Stemming Accuracy: 0.85
